In [ ]:
!pip install "psycopg[binary]==3.3.6" "python-dotenv==1.2.4"


In [ ]:
import os, subprocess, tempfile
import psycopg
from psycopg import sql
from dotenv import load_dotenv

load_dotenv("../../.env")
PGHOST=os.getenv("PGHOST"); PGPORT=os.getenv("PGPORT"); PGUSER=os.getenv("PGUSER")
PGPASSWORD=os.getenv("PGPASSWORD") or None; PGDATABASE=os.getenv("PGDATABASE"); PGSSLMODE=os.getenv("PGSSLMODE")
conn_args=dict(host=PGHOST,port=PGPORT,user=PGUSER,password=PGPASSWORD,sslmode=PGSSLMODE)

with psycopg.connect(autocommit=True,**{**conn_args,"dbname":"postgres"}) as admin:
    if not admin.execute("SELECT 1 FROM pg_database WHERE datname=%s",(PGDATABASE,)).fetchone():
        admin.execute(sql.SQL("CREATE DATABASE {}").format(sql.Identifier(PGDATABASE)))
conn = psycopg.connect(autocommit=True,dbname=PGDATABASE,**conn_args)
print("connected")


In [ ]:
def show(q,p=None):
    with conn.cursor() as cur:
        cur.execute(q,p)
        if cur.description is None:
            print(cur.statusmessage); return
        cols=[d.name for d in cur.description]; rows=cur.fetchall()
    cells=[["NULL" if v is None else str(v) for v in r] for r in rows]
    widths=[max([len(c)]+[len(r[i]) for r in cells]) for i,c in enumerate(cols)]
    print(" | ".join(c.center(w) for c,w in zip(cols,widths)).rstrip())
    print("-+-".join("-"*w for w in widths))
    for r in cells: print(" | ".join(v.ljust(w) for v,w in zip(r,widths)).rstrip())
    print(f"({len(rows)} row{'' if len(rows)==1 else 's'})")


In [ ]:
show("SELECT * FROM pg_stat_bgwriter")


- **What happens:** check WAL/replication settings and checkpoint stats.
- **Why:** know your durability/replication capacity.


In [ ]:
env = os.environ.copy()
if PGPASSWORD: env["PGPASSWORD"]=PGPASSWORD
fd, path = tempfile.mkstemp(suffix=".sql", prefix="backup_demo_")
os.close(fd)
cmd = ["pg_dump", "-h", PGHOST, "-p", str(PGPORT), "-U", PGUSER, "-d", PGDATABASE, "-f", path]
res = subprocess.run(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
print("pg_dump returncode:", res.returncode)
print("backup size:", os.path.getsize(path), "bytes")
os.remove(path)
print("cleaned temp backup")


- **What happens:** dump schema+data to a SQL file using `pg_dump`.
- **Why:** logical backups are portable and great for migrations/testing restores.


In [ ]:
show("SELECT pid, application_name, state, sync_state, write_lag, flush_lag, replay_lag FROM pg_stat_replication")
show("SELECT slot_name, plugin, slot_type, active, restart_lsn FROM pg_replication_slots")


- **What happens:** check if any standbys connected or slots exist.
- **Why:** monitor replication health.


In [ ]:
show("SELECT pg_current_wal_lsn() AS current_lsn, pg_walfile_name(pg_current_wal_lsn()) AS wal_file")


- **What happens:** see current WAL position.
- **Why:** needed for base backups/PITR planning.


In [ ]:
conn.close()
print("done")
